<a href="https://colab.research.google.com/github/Serajummunira/ip-intelligence-enrichment/blob/main/ip_enrichment_pipeline_rdap_geoip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas requests

In [ ]:
import pandas as pd
import requests
import socket
import os
import time

In [ ]:
# Optional (recommended) Set api kEY
os.environ["7f441d77948e77IPINFO_TOKEN"] = "YOUR_IPINFO_TOKEN"
os.environ["599f6e4ac3ef8d08d7d3dd73d35f38dff9c6d295f57693244cd1831a284632664a18c5a231c09bfd"] = "YOUR_ABUSEIPDB_API_KEY"

In [ ]:
#Upload CSV file
from google.colab import files
uploaded = files.upload()

Saving 10000_ip_addresses.csv to 10000_ip_addresses (1).csv


ARIN RDAP (ownership data)

In [ ]:
def get_arin_data(ip):
    try:
        url = f"https://rdap.arin.net/registry/ip/{ip}"
        res = requests.get(url, timeout=10)
        if res.status_code != 200:
            return {}

        data = res.json()

        result = {
            "arin_handle": data.get("handle"),
            "arin_name": data.get("name"),
            "arin_type": data.get("type"),
            "arin_start_ip": data.get("startAddress"),
            "arin_end_ip": data.get("endAddress"),
            "arin_country": data.get("country"),
            "arin_parent": data.get("parentHandle")
        }

        # Extract abuse email if available
        abuse_email = None
        for ent in data.get("entities", []):
            roles = ent.get("roles", [])
            if "abuse" in roles:
                vcard = ent.get("vcardArray", [])
                if len(vcard) > 1:
                    for item in vcard[1]:
                        if item[0] == "email":
                            abuse_email = item[3]

        result["arin_abuse_email"] = abuse_email
        return result

    except:
        return {}

IPinfo (Geo + ASN + ISP)

In [ ]:
def get_ipinfo_data(ip):
    try:
        token = os.getenv("7f441d77948e77")
        url = f"https://ipinfo.io/{ip}/json"
        if token:
            url += f"?token={token}"

        res = requests.get(url, timeout=10)
        if res.status_code != 200:
            return {}

        data = res.json()

        return {
            "geo_country": data.get("country"),
            "geo_region": data.get("region"),
            "geo_city": data.get("city"),
            "geo_location": data.get("loc"),
            "org": data.get("org"),
            "hostname": data.get("hostname")
        }

    except:
        return {}

AbuseIPDB (threat intelligence)

In [ ]:
def get_abuseipdb_data(ip):
    try:
        api_key = os.getenv("599f6e4ac3ef8d08d7d3dd73d35f38dff9c6d295f57693244cd1831a284632664a18c5a231c09bfd")
        if not api_key:
            return {}

        url = "https://api.abuseipdb.com/api/v2/check"
        headers = {
            "Key": api_key,
            "Accept": "application/json"
        }
        params = {
            "ipAddress": ip,
            "maxAgeInDays": 90
        }

        res = requests.get(url, headers=headers, params=params, timeout=10)
        if res.status_code != 200:
            return {}

        data = res.json().get("data", {})

        return {
            "abuse_score": data.get("abuseConfidenceScore"),
            "total_reports": data.get("totalReports"),
            "isp": data.get("isp"),
            "usage_type": data.get("usageType")
        }

    except:
        return {}

Reverse DNS

In [ ]:
def get_reverse_dns(ip):
    try:
        return socket.gethostbyaddr(ip)[0]
    except:
        return None

Run enrichment

In [ ]:
df = pd.read_csv("/content/10000_ip_addresses.csv")

results = []

for ip in df["ip"]:
    print(f"Processing {ip}...")

    record = {"ip": ip}

    # Merge all data sources
    record.update(get_arin_data(ip))
    record.update(get_ipinfo_data(ip))
    record.update(get_abuseipdb_data(ip))

    record["reverse_dns"] = get_reverse_dns(ip)

    results.append(record)

    time.sleep(1)  # avoid rate limits

enriched_df = pd.DataFrame(results)
enriched_df.head()

Streaming output truncated to the last 5000 lines.
Processing 36.137.121.41...
Processing 107.150.105.57...
Processing 172.245.112.205...
Processing 170.106.180.139...
Processing 91.231.89.211...
Processing 65.49.1.115...
Processing 66.132.186.243...
Processing 2.56.126.190...
Processing 64.62.156.27...
Processing 64.62.156.188...
Processing 65.49.1.212...
Processing 178.62.55.235...
Processing 65.49.1.76...
Processing 86.54.31.38...
Processing 43.228.157.191...
Processing 43.106.125.171...
Processing 193.163.125.194...
Processing 184.105.139.103...
Processing 65.49.1.125...
Processing 47.245.138.151...
Processing 193.163.125.127...
Processing 66.132.172.236...
Processing 43.165.185.71...
Processing 192.81.129.137...
Processing 147.182.209.206...
Processing 192.81.129.30...
Processing 151.69.244.7...
Processing 109.105.209.22...
Processing 74.82.47.27...
Processing 65.49.1.186...
Processing 20.14.72.151...
Processing 193.163.125.47...
Processing 47.84.143.228...
Processing 8.216.16.62.

,ip,arin_handle,arin_name,arin_type,arin_start_ip,arin_end_ip,arin_country,arin_parent,arin_abuse_email,geo_country,geo_region,geo_city,geo_location,org,hostname,reverse_dns
0,211.21.61.246,211.21.0.0 - 211.21.255.255,HINET-NET,ASSIGNED PORTABLE,211.21.0.0,211.21.255.255,TW,None,abuse@hinet.net,TW,Taiwan,Taipei,"25.0531,121.5264",AS3462 Data Communication Business Group,211-21-61-246.hinet-ip.hinet.net,211-21-61-246.hinet-ip.hinet.net
1,45.148.10.151,45.148.10.0 - 45.148.10.255,DMZHOST,ASSIGNED PA,45.148.10.0,45.148.10.255,AD,45.148.8.0 - 45.148.11.255,dmzhostabuse@gmail.com,NL,North Holland,Amsterdam,"52.3740,4.8897",AS48090 TECHOFF SRV LIMITED,None,None
2,85.217.140.53,85.217.140.0 - 85.217.140.255,NL-MODAT-20050118,ALLOCATED PA,85.217.140.0,85.217.140.255,FR,0.0.0.0 - 255.255.255.255,abuse@modat.io,FR,Hauts-de-France,Calais,"50.9519,1.8563",AS209334 Modat B.V.,o352.scanner.modat.io,o352.scanner.modat.io
3,220.119.37.141,220.116.0.0 - 220.119.255.255,KORNET-KR,ALLOCATED PORTABLE,220.116.0.0,220.119.255.255,KR,None,hostmaster@nic.or.kr,KR,Gyeongsangnam-do,Changwon,"35.2281,128.6811",AS4766 Korea Telecom,None,None
4,2.57.121.118,2.57.121.0 - 2.57.121.255,UNMANAGED-LTD,ASSIGNED PA,2.57.121.0,2.57.121.255,GB,2.57.120.0 - 2.57.123.255,abuse@unmanaged.uk,RO,Timiș County,Timişoara,"45.7537,21.2257",AS47890 UNMANAGED LTD,dns118.personaliseplus.com,dns118.personaliseplus.com


Save output

In [ ]:
enriched_df.to_csv("geoip_10000_output.csv", index=False)

from google.colab import files
files.download("geoip_10000_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>